# BTC Oracle: CNN-LSTM Hybrid

**CNN** extracts local patterns + **LSTM** captures temporal dependencies

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
print(f'TensorFlow: {tf.__version__}')

## 1. Load Data

In [ ]:
path = kagglehub.dataset_download('oussamataghlaoui/btc-oracle-on-chain-sentiment-and-macro-data')
import glob
csv_files = glob.glob(path + '/*.csv')
df = pd.read_csv(csv_files[0], parse_dates=['Datetime'], index_col='Datetime')
print(f'Shape: {df.shape}')

## 2. Prepare Data (Same as GRU)

In [ ]:
# Create target
def create_target(df, percentile=60):
    returns = df['future_return_24h']
    up = np.percentile(returns.dropna(), percentile)
    down = np.percentile(returns.dropna(), 100-percentile)
    target = pd.Series(1, index=returns.index)
    target[returns > up] = 2
    target[returns < down] = 0
    return target.dropna()

df_clean = df.dropna(subset=['future_return_24h']).copy()
y = create_target(df_clean)

# Features
feature_cols = [c for c in df_clean.columns if 'future' not in c and 'target' not in c]
feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df_clean[c])]
X = df_clean[feature_cols].replace([np.inf, -np.inf], np.nan)

# Split & Scale
train_size = int(len(X) * 0.7)
val_size = int(len(X) * 0.15)

X_train = X.iloc[:train_size].fillna(X.iloc[:train_size].median())
X_val = X.iloc[train_size:train_size+val_size].fillna(X.iloc[:train_size].median())
X_test = X.iloc[train_size+val_size:].fillna(X.iloc[:train_size].median())

y_train = y.iloc[:train_size]
y_val = y.iloc[train_size:train_size+val_size]
y_test = y.iloc[train_size+val_size:]

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Sequences
def create_sequences(X, y, seq_len=48):
    Xs, ys = [], []
    for i in range(len(X) - seq_len):
        Xs.append(X[i:i+seq_len])
        ys.append(y.iloc[i+seq_len])
    return np.array(Xs), np.array(ys)

X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train)
X_val_seq, y_val_seq = create_sequences(X_val_scaled, y_val)
X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_test)

print(f'Train: {X_train_seq.shape}')

## 3. Build CNN-LSTM Model

In [ ]:
model = tf.keras.Sequential([
    # CNN layers
    tf.keras.layers.Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(48, X_train_seq.shape[2])),
    tf.keras.layers.MaxPooling1D(pool_size=2),
    tf.keras.layers.Conv1D(filters=32, kernel_size=3, activation='relu'),
    tf.keras.layers.MaxPooling1D(pool_size=2),
    
    # LSTM layers
    tf.keras.layers.LSTM(50, return_sequences=True),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.LSTM(25),
    tf.keras.layers.Dropout(0.3),
    
    # Output
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 4. Train

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    X_train_seq, y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=50,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

## 5. Evaluate

In [ ]:
# Curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history.history['loss'], label='Train')
ax1.plot(history.history['val_loss'], label='Val')
ax1.set_title('Loss')
ax1.legend()

ax2.plot(history.history['accuracy'], label='Train')
ax2.plot(history.history['val_accuracy'], label='Val')
ax2.set_title('Accuracy')
ax2.legend()
plt.show()

In [ ]:
# Predictions
y_prob = model.predict(X_test_seq)
y_pred = np.argmax(y_prob, axis=1)
y_conf = np.max(y_prob, axis=1)

print('--- Standard Report ---')
print(classification_report(y_test_seq, y_pred, target_names=['DOWN', 'NEUTRAL', 'UP']))

# High Confidence
hc_mask = y_conf >= 0.75
if np.sum(hc_mask) > 0:
    print(f'\n--- High Confidence (>75%) ---')
    print(f'Coverage: {np.sum(hc_mask)/len(y_test_seq):.1%}')
    print(classification_report(y_test_seq[hc_mask], y_pred[hc_mask], target_names=['DOWN', 'NEUTRAL', 'UP']))
    
    plt.figure(figsize=(6, 5))
    sns.heatmap(confusion_matrix(y_test_seq[hc_mask], y_pred[hc_mask]), annot=True, fmt='d', cmap='Blues')
    plt.title('High Confidence Confusion Matrix')
    plt.show()

## 6. Save

In [ ]:
model.save('../models/cnn_lstm_model.keras')
print('✓ Saved')